# V-NLI aplicado para la detección de contenido NSFW y Armas

El presente Notebook tiene como cometido utilizar la arquitectura V-NLI(Visual Natural Language Interface). Está es una arquitectura de interfaz interactiva que combina el procesamiento de lenguaje natural (NLP) con componentes visuales para hacer los sistemas de machine learning más accesibles y explicables.

Una de las razones más predominantes para elegir está arquitectura fue que, gracias a que se pueden hacer inferencias no necesita de entrenamiento con datasets, sino solo tener varias hipótesis que permitan validar lo que se ve contra lo que se describe.

El modelo a utilizar es el ***MobileCLIP*** porque es uno de lo más rápidos para dispositivos móviles y ya viene preentrenado.

## Instalación

In [1]:
# Instalar la librería desde el repo oficial
!git clone https://github.com/apple/ml-mobileclip.git
%cd ml-mobileclip
!pip install -e . -q

Cloning into 'ml-mobileclip'...
remote: Enumerating objects: 235, done.
remote: Counting objects: 100% (96/96), done.
remote: Compressing objects: 100% (65/65), done.
remote: Total 235 (delta 47), reused 33 (delta 31), pack-reused 139 (from 1)
Receiving objects: 100% (235/235), 4.46 MiB | 17.18 MiB/s, done.
Resolving deltas: 100% (93/93), done.
/content/ml-mobileclip
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.3/104.3 MB 8.4 MB/s eta 0:00:00


In [2]:
# Descargar solo un modelo (ej: S0, el más ligero)
!pip install huggingface_hub -q
from huggingface_hub import hf_hub_download

# Modelos disponibles: MobileCLIP-S0, S1, S2, B, B-LT, S3, L-14, S4
ckpt_path = hf_hub_download(
    repo_id="apple/MobileCLIP-S1",
    filename="mobileclip_s1.pt"
)
print(f"Checkpoint guardado en: {ckpt_path}")

mobileclip_s1.pt:   0%|          | 0.00/341M [00:00<?, ?B/s]

Checkpoint guardado en: /root/.cache/huggingface/hub/models--apple--MobileCLIP-S1/snapshots/9e9005f93d5d8eb197aac25cf53af31364ccd489/mobileclip_s1.pt


## Realizar inferencias
El siguiente código es un ejemplo de prueba para poder entender como funciona y como clasifica ***MobileCLIP***

In [3]:
# Cargar modelo y hacer inferencia
import torch
from PIL import Image
import mobileclip
import requests
from io import BytesIO

model, _, preprocess = mobileclip.create_model_and_transforms(
    'mobileclip_s1',
    pretrained=ckpt_path  # ruta al .pt descargado
)
tokenizer = mobileclip.get_tokenizer('mobileclip_s1')

model.eval()

# Cargar imagen de ejemplo desde el /content
url = "/content/perro_train.jpeg"
image = preprocess(Image.open(url).convert('RGB')).unsqueeze(0)
text = tokenizer(["a diagram", "a dog", "a cat", "a ball"])

with torch.no_grad(), torch.cuda.amp.autocast():
    image_features = model.encode_image(image)
    text_features = model.encode_text(text)
    image_features /= image_features.norm(dim=-1, keepdim=True)
    text_features /= text_features.norm(dim=-1, keepdim=True)
    text_probs = (100.0 * image_features @ text_features.T).softmax(dim=-1)

print("Label probs:", text_probs) # Obtenemos un vector de probabilidad donde la posición más cercana a 1 es la clase más probable

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/tmp/ipykernel_12763/3833123478.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast():
/usr/local/lib/python3.12/dist-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(


Label probs: tensor([[8.8750e-07, 9.9981e-01, 5.3318e-07, 1.9178e-04]])


## Ejemplo de inferencia 2
El siguiente ejemplo es más completo porque muestra la clasificación de 2 imágenes seguras(perro y manzana) contra 1 insegura(arma).

Usualmente, funcionan con una premisa (una afirmación base) y una hipótesis (la afirmación que se quiere evaluar).

Las redes neuronales analizan ambos textos y clasifican la relación en tres categorías principales:

- Entailment (Implicación): Si la premisa es verdadera, la hipótesis también debe serlo por lógica.

- Contradiction (Contradicción): Si la premisa es verdadera, la hipótesis es necesariamente falsa.

- Neutral: La premisa no proporciona suficiente información para confirmar o negar la hipótesis.

In [13]:
import torch
import mobileclip
from PIL import Image
import requests
from io import BytesIO
import os
import numpy as np


# 1. Configuración del Modelo
model_name = "mobileclip_s1"

model, _, preprocess = mobileclip.create_model_and_transforms(
    model_name,
    pretrained=ckpt_path
)
tokenizer = mobileclip.get_tokenizer(model_name)
model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)


# 2. Hipótesis aceptada(riesgo) y contradictoria(seguridad)
categorias_riesgo = [
    "a photo of illegal drugs, syringes, or explicit drug consumption",
    "an explicit photo of nudity, pornography, or sexual acts",
    "a clear photograph of firearms, handguns, rifles, or weapons",
    "an image depicting narco-culture, drug cartels, and criminal activities",
    "a graphic image of physical violence, fighting, or assault",
    "a photo of a dead body, corpse, or explicit death",
    "a sensationalist image with blood, severe injuries, and gore"
]

categorias_seguras = [
    "a photo of a dog, cat, or a pet in a normal setting",
    "a photo of fruit, food, or everyday household objects",
    "a beautiful landscape, nature, or outdoor environment",
    "a close-up of plants, flowers, or wildlife",
    "a photo of happy people in a safe, everyday social situation",
    "a clean, high-quality, safe, and normal photograph"
]

todas_las_categorias = categorias_riesgo + categorias_seguras

# 3. Si se necesita subir una imagen por url para cargar imagen(solo funciona si el sitio no bloquea la extracción)
def cargar_imagen(path_o_url):
    if path_o_url.startswith(("http://", "https://")):
        response = requests.get(path_o_url, timeout=15)
        response.raise_for_status()
        return Image.open(BytesIO(response.content)).convert("RGB")
    else:
        if not os.path.exists(path_o_url):
            raise FileNotFoundError(f"No existe el archivo: {path_o_url}")
        return Image.open(path_o_url).convert("RGB")


# 4. Función principal
def moderar_imagen(path_o_url, umbral_riesgo=0.55, margen_seguridad=0.10, debug=False):
    try:
        img_raw = cargar_imagen(path_o_url)
    except Exception as e:
        return f"ERROR al cargar imagen: {e}"

    image = preprocess(img_raw).unsqueeze(0).to(device) # Esta es la imagen que se va a comparar
    text = tokenizer(todas_las_categorias).to(device) # Estos son las hipótesis contra las que la imagen se compara

    with torch.no_grad(): # Desactiva el cálculo de gradientes. No se está entrenando el modelo solo usándolo
        if device == "cuda":
            with torch.cuda.amp.autocast():
                image_features = model.encode_image(image)
                text_features = model.encode_text(text)
        else:
            image_features = model.encode_image(image)
            text_features = model.encode_text(text)

        # Las siguientes líneas ayudan a saber la similitud entre la imagen y el texto
        image_features /= image_features.norm(dim=-1, keepdim=True)
        text_features /= text_features.norm(dim=-1, keepdim=True)

        probs = (100.0 * image_features @ text_features.T).softmax(dim=-1)
        probs = probs.squeeze().cpu().numpy()

    n_riesgo = len(categorias_riesgo)
    probs_riesgo = probs[:n_riesgo]
    probs_seguras = probs[n_riesgo:]

    idx_riesgo = int(np.argmax(probs_riesgo))
    idx_segura = int(np.argmax(probs_seguras))

    mejor_riesgo = float(probs_riesgo[idx_riesgo])
    mejor_segura = float(probs_seguras[idx_segura])

    etiqueta_riesgo = categorias_riesgo[idx_riesgo]
    etiqueta_segura = categorias_seguras[idx_segura]

    # Regla 1: si una clase segura gana entonces, permitir
    if mejor_segura >= mejor_riesgo + margen_seguridad:
        if debug:
            return {
                "decision": "PERMITIR",
                "mejor_segura": (etiqueta_segura, round(mejor_segura, 4)),
                "mejor_riesgo": (etiqueta_riesgo, round(mejor_riesgo, 4)),
                "probs": {cat: round(float(p), 4) for cat, p in zip(todas_las_categorias, probs)}
            }
        return f"PERMITIR (segura={mejor_segura:.2f}, riesgo={mejor_riesgo:.2f})"

    # Regla 2: si una clase de riesgo gana entonces, bloquear
    if mejor_riesgo >= umbral_riesgo and mejor_riesgo > mejor_segura:
        if debug:
            return {
                "decision": "BLOQUEAR",
                "mejor_riesgo": (etiqueta_riesgo, round(mejor_riesgo, 4)),
                "mejor_segura": (etiqueta_segura, round(mejor_segura, 4)),
                "probs": {cat: round(float(p), 4) for cat, p in zip(todas_las_categorias, probs)}
            }
        return f"BLOQUEAR ({etiqueta_riesgo[:45]}..., score={mejor_riesgo:.2f})"

    # Regla 3: caso ambiguo, mejor permitir con revisión o tratarlo aparte
    if debug:
        return {
            "decision": "REVISAR",
            "mejor_riesgo": (etiqueta_riesgo, round(mejor_riesgo, 4)),
            "mejor_segura": (etiqueta_segura, round(mejor_segura, 4)),
            "probs": {cat: round(float(p), 4) for cat, p in zip(todas_las_categorias, probs)}
        }
    return f"REVISAR (segura={mejor_segura:.2f}, riesgo={mejor_riesgo:.2f})"


# 5. Pruebas
print("Perro:", moderar_imagen("/content/perro_train.jpeg", debug=True))
print("Manzana:", moderar_imagen("/content/apple_train.jpeg", debug=True))
print("Arma:", moderar_imagen("/content/arma_train.jpeg", debug=True))

Perro: {'decision': 'PERMITIR', 'mejor_segura': ('a photo of a dog, cat, or a pet in a normal setting', 0.9993), 'mejor_riesgo': ('a photo of a dead body, corpse, or explicit death', 0.0), 'probs': {'a photo of illegal drugs, syringes, or explicit drug consumption': 0.0, 'an explicit photo of nudity, pornography, or sexual acts': 0.0, 'a clear photograph of firearms, handguns, rifles, or weapons': 0.0, 'an image depicting narco-culture, drug cartels, and criminal activities': 0.0, 'a graphic image of physical violence, fighting, or assault': 0.0, 'a photo of a dead body, corpse, or explicit death': 0.0, 'a sensationalist image with blood, severe injuries, and gore': 0.0, 'a photo of a dog, cat, or a pet in a normal setting': 0.9993, 'a photo of fruit, food, or everyday household objects': 0.0, 'a beautiful landscape, nature, or outdoor environment': 0.0003, 'a close-up of plants, flowers, or wildlife': 0.0, 'a photo of happy people in a safe, everyday social situation': 0.0, 'a clean, 

## Evaluación de contenido NSFW
El siguiente dataset importado es de contenido NSFW, lo queremos analizar para poder contrastar nuestro modelo anterior contra este contenido y saber cuanto bloquea realmente. Para ello, primero necesitamos saber de que clases está compuesto el dataset y hacer una mezcla más pequea de este para poder evaluar más rápido el modelo.

In [5]:
from datasets import load_dataset

# 1. Cargamos la información del dataset (metadata) sin descargar los datos aún
print(" Explorando metadatos del dataset...")
dataset_info = load_dataset("deepghs/nsfw_detect", split="train", streaming=True)

# 2. Ver la estructura de las columnas (Features)
print("\n*** ESTRUCTURA DE COLUMNAS (FEATURES) ***")
print(dataset_info.features)

# 3. Extraer los nombres de las etiquetas (Labels)
# Esto nos dirá qué significa 0 y qué significa 1
label_names = dataset_info.features['label'].names
print(f"\n*** MAPA DE ETIQUETAS ***")
for i, name in enumerate(label_names):
    print(f"ID {i}: {name}")

dataset_mezclado = dataset_info.shuffle(seed=42, buffer_size=20000)

# 4. Ver una muestra real (Head manual)
print("\n*** INSPECCIÓN DE LOS PRIMEROS 3 ELEMENTOS ***")
# Tomamos una pequeña muestra para ver el contenido de los diccionarios
iterador = iter(dataset_mezclado)
for _ in range(5):
    muestra = next(iterador)
    # No imprimimos la imagen completa porque son miles de números (pixeles)
    # Solo imprimimos el tamaño de la imagen y su etiqueta
    print(f"Imagen: {muestra['image'].size} px | Etiqueta (ID): {muestra['label']} ({label_names[muestra['label']]})")


 Explorando metadatos del dataset...


README.md: 0.00B [00:00, ?B/s]


*** ESTRUCTURA DE COLUMNAS (FEATURES) ***
{'image': Image(mode=None, decode=True), 'label': ClassLabel(names=['drawings', 'hentai', 'neutral', 'porn', 'sexy'])}

*** MAPA DE ETIQUETAS ***
ID 0: drawings
ID 1: hentai
ID 2: neutral
ID 3: porn
ID 4: sexy

*** INSPECCIÓN DE LOS PRIMEROS 3 ELEMENTOS ***
Imagen: (1066, 600) px | Etiqueta (ID): 0 (drawings)
Imagen: (600, 901) px | Etiqueta (ID): 2 (neutral)
Imagen: (800, 600) px | Etiqueta (ID): 2 (neutral)
Imagen: (600, 459) px | Etiqueta (ID): 1 (hentai)
Imagen: (600, 945) px | Etiqueta (ID): 1 (hentai)


In [6]:
# Definimos un arreglo para guardar el mini dataset
mini_dataset = []
limite = 100

# Usamos el dataset_mezclado que del bloque anterior
iterador = iter(dataset_mezclado)

for i in range(limite):
    try:
        # Obtenemos la muestra (esto descarga la imagen en ese momento)
        muestra = next(iterador)

        # Guardamos un diccionario limpio
        mini_dataset.append({
            'imagen': muestra['image'].convert("RGB"),
            'label_id': muestra['label'],
            'label_name': label_names[muestra['label']]
        })

        if (i + 1) % 20 == 0:
            print(f"Cargadas {i + 1}/{limite} muestras...")

    except StopIteration:
        print("Fin del dataset alcanzado antes de llegar al límite.")
        break

print(f"\nProceso completado. Tienes {len(mini_dataset)} imágenes en memoria.")

Cargadas 20/100 muestras...
Cargadas 40/100 muestras...
Cargadas 60/100 muestras...
Cargadas 80/100 muestras...
Cargadas 100/100 muestras...

Proceso completado. Tienes 100 imágenes en memoria.


In [7]:
import torch
import numpy as np
from sklearn.metrics import accuracy_score, classification_report

# 1. Configuración de la lógica
clases_nsfw_id = [1, 3, 4]  # hentai, porn, sexy son categorías de riesgo
umbral_riesgo = 0.40        # Un umbral bajo nos dice que bloquee a patir del 40% de sospecha, entre más lo subimos más permisivos somos
margen_seguridad = 0.10     # El margen de seguridad nos ayuda a que si las probabilidades son casi iguales gane una por lo menos con 10 puntos de margen más que la otra

# Contadores
resultados_por_clase = {name: {'total': 0, 'bloqueadas': 0, 'permitidas': 0} for name in label_names}
true_labels_bin = []
pred_labels_bin = []

print(f"Evaluando MobileCLIP sobre {len(mini_dataset)} muestras con lógica de competencia...")

model.to(device)
n_riesgo = len(categorias_riesgo)

for i, data in enumerate(mini_dataset):
    # Inferencia Completa
    img_t = preprocess(data['imagen']).unsqueeze(0).to(device)
    # IMPORTANTE: Usar TODAS las categorías para que compitan de esta forma compara las probabilidades de cada una
    txt_t = tokenizer(todas_las_categorias).to(device)

    with torch.no_grad():
        with torch.cuda.amp.autocast():
            img_f = model.encode_image(img_t)
            txt_f = model.encode_text(txt_t)

            img_f /= img_f.norm(dim=-1, keepdim=True)
            txt_f /= txt_f.norm(dim=-1, keepdim=True)

            probs = (40.0 * img_f @ txt_f.T).softmax(dim=-1).squeeze().cpu().numpy()

    # Lógica de Decisión
    probs_riesgo = probs[:n_riesgo]
    probs_seguras = probs[n_riesgo:]

    mejor_riesgo = np.max(probs_riesgo) # Toma la probabilidad más alta
    mejor_segura = np.max(probs_seguras)

    # Aplicamos las Reglas 1 y 2
    if mejor_segura >= mejor_riesgo + margen_seguridad:
        bloquear = False
    elif mejor_riesgo >= umbral_riesgo and mejor_riesgo > mejor_segura:
        bloquear = True
    else:
        # Regla 3: Ambigüedad, si el modelo no está seguro preferimos bloquear
        bloquear = False

    label_pred = 1 if bloquear else 0
    label_real_bin = 1 if data['label_id'] in clases_nsfw_id else 0

    # Estadísticas
    nombre_clase = data['label_name']
    resultados_por_clase[nombre_clase]['total'] += 1
    if bloquear:
        resultados_por_clase[nombre_clase]['bloqueadas'] += 1
    else:
        resultados_por_clase[nombre_clase]['permitidas'] += 1

    true_labels_bin.append(label_real_bin)
    pred_labels_bin.append(label_pred)

# Reporte
print("\n" + "="*55)
print(f"{'Categoría':<12} | {'Total':<6} | {'Bloqueadas':<10} | {'Permitidas':<10}")
print("-" * 55)
for clase, stats in resultados_por_clase.items():
    print(f"{clase:<12} | {stats['total']:<6} | {stats['bloqueadas']:<10} | {stats['permitidas']:<10}")

acc = accuracy_score(true_labels_bin, pred_labels_bin)
print("="*55)
print(f"ACCURACY GLOBAL: {acc:.2%}")
print("="*55)

Evaluando MobileCLIP sobre 100 muestras con lógica de competencia...


/tmp/ipykernel_12763/140046405.py:27: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():



Categoría    | Total  | Bloqueadas | Permitidas
-------------------------------------------------------
drawings     | 23     | 7          | 16        
hentai       | 31     | 20         | 11        
neutral      | 33     | 8          | 25        
porn         | 13     | 13         | 0         
sexy         | 0      | 0          | 0         
ACCURACY GLOBAL: 74.00%


Si observamos el bloque de código anterior, nos damos cuenta que el accuracy global es 74%, es el más alto al que se llegó después de alterar el `margen de seguridad`, el `umbral de riesgo` y la `probabilidad`(propbs).

## Evaluación de contenido con armas

Los siguientes bloques de código son para evalaur que tan bueno es el modelo para detectar armas.

In [8]:
import random

# Cargamos el dataset de armas
weapons_ds = load_dataset("Subh775/WeaponDetection", split="train", streaming=True)

weapons_subset = []
# Mezclamos con un buffer grande para asegurar variedad de armas
iterador_weapons = iter(weapons_ds.shuffle(seed=42, buffer_size=2000))

for i in range(50):
    try:
        muestra = next(iterador_weapons)
        # En este dataset, todas las imágenes tienen armas,
        # así que la etiqueta real es siempre 1 (Riesgo)
        weapons_subset.append({
            'imagen': muestra['image'].convert("RGB"),
            'label_binaria': 1,
            'clase_original': "weapon" # Simplificamos ya que el fin es bloquear
        })
    except StopIteration:
        break

# Extraemos 50 imágenes SEGURAS del dataset anterior (mini_dataset)
# Solo tomamos 'neutral' y 'drawings'
seguras_originales = [d for d in mini_dataset if d['label_id'] in [0, 2]]
random.shuffle(seguras_originales)
seguras_subset = []

for d in seguras_originales[:50]:
    seguras_subset.append({
        'imagen': d['imagen'],
        'label_binaria': 0,
        'clase_original': d['label_name']
    })

# Fusión y mezcla final
super_eval_dataset = weapons_subset + seguras_subset
random.shuffle(super_eval_dataset)

print(f"Dataset listo {len(super_eval_dataset)} imágenes (50/50 balanceado).")

README.md: 0.00B [00:00, ?B/s]

Dataset listo 100 imágenes (50/50 balanceado).


In [9]:
import torch
import numpy as np
from sklearn.metrics import accuracy_score, classification_report

# 1. Preparar métricas
true_labels = []
pred_labels = []

# Usamos un diccionario genérico para ver qué tan bien detecta las armas vs las seguras
stats_detalle = {
    "weapon": {"total": 0, "bloqueadas": 0},
    "safe_content": {"total": 0, "bloqueadas": 0}
}

model.to(device)
n_riesgo = len(categorias_riesgo)

for data in super_eval_dataset:
    # Preprocesamiento e Inferencia
    img_t = preprocess(data['imagen']).unsqueeze(0).to(device)
    txt_t = tokenizer(todas_las_categorias).to(device)

    with torch.no_grad():
        if device == "cuda":
            with torch.cuda.amp.autocast():
                img_f = model.encode_image(img_t)
                txt_f = model.encode_text(txt_t)
        else:
            img_f = model.encode_image(img_t)
            txt_f = model.encode_text(txt_t)

        img_f /= img_f.norm(dim=-1, keepdim=True)
        txt_f /= txt_f.norm(dim=-1, keepdim=True)

        # Logit scale de 50 para balancear precisión/sensibilidad
        probs = (50.0 * img_f @ txt_f.T).softmax(dim=-1).squeeze().cpu().numpy()

    # Lógica de Decisión
    prob_riesgo_max = np.max(probs[:n_riesgo])
    prob_seguro_max = np.max(probs[n_riesgo:])

    # Aplicamos el umbral de confianza (0.45) y la competencia con margen (0.05)
    bloquear = (prob_riesgo_max > prob_seguro_max + 0.05) and (prob_riesgo_max > 0.45)

    label_pred = 1 if bloquear else 0
    label_real = data['label_binaria']

    true_labels.append(label_real)
    pred_labels.append(label_pred)

    # Registro de estadísticas
    tipo = "weapon" if label_real == 1 else "safe_content"
    stats_detalle[tipo]["total"] += 1
    if bloquear:
        stats_detalle[tipo]["bloqueadas"] += 1

# 2. Reporte Final
print("\n" + "="*40)
print("REPORTE DE RENDIMIENTO FUSIONADO")
print("="*40)
acc = accuracy_score(true_labels, pred_labels)
print(f"Accuracy Global: {acc:.2%}")

print("\nDesglose por origen:")
# Efectividad detectando armas
w_total = stats_detalle["weapon"]["total"]
w_det = stats_detalle["weapon"]["bloqueadas"]
print(f"- Armas detectadas (Recall): {w_det}/{w_total} ({w_det/w_total:.1%})")

# Evitar falsos positivos en contenido seguro
s_total = stats_detalle["safe_content"]["total"]
s_perm = s_total - stats_detalle["safe_content"]["bloqueadas"]
print(f"- Seguro permitido (Specificity): {s_perm}/{s_total} ({s_perm/s_total:.1%})")

print("\n" + "="*40)
print(classification_report(true_labels, pred_labels, target_names=['SFW (Permitir)', 'NSFW (Bloquear)']))


REPORTE DE RENDIMIENTO FUSIONADO
Accuracy Global: 73.00%

Desglose por origen:
- Armas detectadas (Recall): 38/50 (76.0%)
- Seguro permitido (Specificity): 35/50 (70.0%)

                 precision    recall  f1-score   support

 SFW (Permitir)       0.74      0.70      0.72        50
NSFW (Bloquear)       0.72      0.76      0.74        50

       accuracy                           0.73       100
      macro avg       0.73      0.73      0.73       100
   weighted avg       0.73      0.73      0.73       100



Si observamos el modelo anterior, nos damos cuenta que el accuracy global es 73%, es el más alto al que se llegó después de alterar el `margen de seguridad`, el `umbral de riesgo` y la `probabilidad`(propbs).

## Exportación del modelo a Onnx

En el siguiente bloque de código exportamoe el modelo en formato `.onnx` para que sea lijero y podamos tenerlo en dispositivos móviles.

In [10]:
import torch
import os

# (Forzar Opset 17 para evitar el error de LayerNormalization)
OPSET = 17

print("Exportando Image Encoder...")
try:
    torch.onnx.export(
        model.image_encoder,
        dummy_image,
        "mobileclip_image.onnx",
        export_params=True,
        opset_version=OPSET,
        do_constant_folding=True,
        input_names=['image'],
        output_names=['image_features']
    )
    print("Image Encoder guardado.")
except Exception as e:
    print(f"Error en Image: {e}")

print("Exportando Text Encoder...")
try:
    torch.onnx.export(
        model.text_encoder,
        dummy_text,
        "mobileclip_text.onnx",
        export_params=True,
        opset_version=OPSET,
        do_constant_folding=True,
        input_names=['text'],
        output_names=['text_features']
    )
    print("Text Encoder guardado.")
except Exception as e:
    print(f"Error en Text: {e}")

# Verificamos físicamente
print("\nArchivos en el directorio actual:")
print([f for f in os.listdir('.') if f.endswith(('.onnx', '.npy'))])

Exportando Image Encoder...
Error en Image: name 'dummy_image' is not defined
Exportando Text Encoder...
Error en Text: name 'dummy_text' is not defined

Archivos en el directorio actual:
[]


Es importante que guardemos los textos(hipótesis) porque estas ayudan a nuestro modelo a tener el accuracy anterior.

In [11]:
import numpy as np

# Generamos las features de texto con el modelo actual
text_tokens = tokenizer(todas_las_categorias).to(device)
with torch.no_grad():
    text_features = model.encode_text(text_tokens)
    text_features /= text_features.norm(dim=-1, keepdim=True)

# Guardamos esto como un archivo .npy
# Esto contiene "el alma" de tu lógica de moderación
np.save("text_features_anchors.npy", text_features.cpu().numpy())
print("Características de texto guardadas! Terminamos :3")

Características de texto guardadas! Terminamos :3


# Gracias por llegar hasta aquí

Si leíste todo mi código y te pareció bueno sígueme en mis redes como:
- ***Yael-PM*** GITHUB
- ***yael__pm*** IG
- ***yaelpm.dev08@gmail.com*** GMAIL

Si encuentras algún error o sugerencia de igual forma hazmelo saber

                :                
               ;                
              :                 
              ;      Yael PM.           
             /                  
           o/                   
         ._/\___,                
             \                  
             /                   
             `      